In [51]:
#Exercise 12 from hands on machine learning chapter 4  - Pytorch edition

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.datasets import load_iris
iris = load_iris(as_frame = True)


In [52]:
iris.data.head(4)

#So given this data, let's split into training and test data


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2


In [53]:
X = iris.data[[ "petal length (cm)", "petal width (cm)", "sepal length (cm)", "sepal width (cm)"]]
y = iris["target"].values

X_bias = np.c_[np.ones(len(X)), X]   #dummy term

In [54]:
#splitting the data into training, test, and validation
test_ratio = 0.2
validation_ratio = 0.2

total_size = len(X_bias)

test_size = int(total_size * test_ratio)
val_size = int(total_size * validation_ratio)

train_size = total_size - test_size - val_size
#now random indices for the split
rng = np.random.default_rng(seed = 42)
rnd_indices = rng.permutation(total_size)  # shuffled array of random indices

X_train = X_bias[rnd_indices[:train_size]]
y_train = y[rnd_indices[:train_size]]

X_valid = X_bias[rnd_indices[train_size:-test_size]]
y_valid = y[rnd_indices[train_size:-test_size]]

X_test = X_bias[rnd_indices[-test_size:]]
y_test = y[rnd_indices[-test_size:]]


In [55]:

def to_one_hot(y):   #one hot implementation without np that I did for fun
    n_classes = y.max() + 1
    m = len(y)
    Y_one_hot = []
    for i in range(m):
        row = []
        for j in range(n_classes):
            if j == y[i]:
                row.append(1.0)
            else:
                row.append(0.0)
        Y_one_hot.append(row)
    return Y_one_hot


In [56]:
def softmax(logits):  #logits is score before softmax
    exps = np.exp(logits)
    exp_sums = np.sum(exps, axis = 1, keepdims = True)
    return exps / exp_sums

n_inputs = X_train.shape[1]
n_outputs = len(np.unique(y_train))
m = len(X_train)

#this is the model weights
np.random.seed(42)
Theta = np.random.randn(n_inputs, n_outputs) # so this makes a 5x3 matrix since there are different weights for each class
#random theta which will change based on the gradient descent
best_theta = None

In [57]:
#this is the model weights
np.random.seed(10)
Theta = np.random.randn(n_inputs, n_outputs) # so this makes a 5x3 matrix since there are different weights for each class
#random theta which will change based on the gradient descent

eta = 0.5
n_epochs = 10000
m = len(X_train)
epsilon = 1e-5

best_loss = np.inf
epochs_without_improvement = 0
patience = 3000

for epoch in range(n_epochs):
    logits = X_train @ Theta
    Y_probs = softmax(logits)

    Y_train_one_hot = to_one_hot(y_train)  #one hot encoding the labels
    
    loss = -np.mean(np.sum(Y_train_one_hot * np.log(Y_probs + epsilon), axis=1))  #mininmizing this using gradient descent
    
    error = Y_probs - Y_train_one_hot
    gradients = (1/m )* X_train.T@error   #this is a matrix containing the partial derivative with respect to each weight from each class
    #so 5x3 in this case

    Theta = Theta - eta *gradients  #changing theta to approach a minimum for loss function

    if loss< 1:
        eta = 0.01
    else:
        eta = 0.1
    if epoch % 500 == 0:
        print(epoch, loss)
        
    if loss< best_loss:
        best_loss = loss
        best_theta = Theta.copy()  #seperate copy instead of pointer towards theta
    else:
        epochs_without_improvement += 1
    if epochs_without_improvement > patience:   #if the model runs for 3000 epochs without improving, automatically stop
        print(epoch , best_loss, "early stopping")
        break
    
        


0 5.114458619665946
500 0.4642270021102319
1000 0.3694375114536719
1500 0.31194037232701455
2000 0.27248228275588204
2500 0.24369190490361597
3000 0.2217586763625066
3500 0.20448344145043318
4000 0.19051029290773608
4500 0.17896049546563295
5000 0.1692409757630614
5500 0.16093762037095305
6000 0.15375279329386182
6500 0.14746718914963228
7000 0.14191569360957393
7500 0.136971623151469
8000 0.13253614919360712
8500 0.12853102668181873
9000 0.12489348530375616
9500 0.1215725699053678


In [58]:
# 96.6% accuracy on the validation set
logits_valid = X_valid @ best_theta
Y_probs_valid = softmax(logits_valid)
Y_predict_valid = np.argmax(Y_probs_valid, axis = 1)
accuracy_valid = np.sum(Y_predict_valid == y_valid)/len(y_valid)
print(accuracy_valid)


0.9666666666666667


In [59]:
#100% accuracy on test set
logits_test = X_test @ best_theta
Y_probs_test = softmax(logits_test)
Y_predict_test = np.argmax(Y_probs_test, axis = 1)
accuracy_test = np.sum(Y_predict_test == y_test)/len(y_test)
print(accuracy_test)

1.0
